# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duashakeel0/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding #4 — "The Freshness Multiplier"** (page 9): "365+ day content that was refreshed within 30 days shows 3.2x health boost (from 10.7 to 34.5) and 57x more impressions (from 71 to 4039)." This is presented as one of the paper's strongest levers and becomes Priority Action #1 in the playbook.

*Where does the label come from?* Two populations of old (365+ day) content: recently-refreshed vs. not. Not a predicted label, an observed group split.

*Does the validation design carry the claim?* This is the question I can't answer from the paper alone, and it matters a lot here: is this a **paired before/after** comparison on the *same* pages (refresh caused the lift), or a **cross-sectional** comparison between two *different* populations of old pages (refreshed vs. never-refreshed)? If it's cross-sectional, there's an obvious confound the paper doesn't address: editors don't refresh pages at random -- they tend to pick pages that already show promise, existing demand, or strategic importance. That selection itself could produce most of the gap, with no causal refresh effect at all. This is structurally the same shape as the "decision-derived feature" trap from the leakage taxonomy: a page getting refreshed is itself a decision someone already made, correlated with what that person already believed about the page. The paper's own Methodology section says "correlations do not prove causation," which is the right instinct -- but Finding #4's language ("one of the strongest measured levers available") reads more causal than that disclaimer fully walks back, and Myth #7 elsewhere in the same paper ("freshness amplifies quality, it does not replace it") suggests the authors know quality-selection matters, without applying that lens back to Finding #4 specifically.

**ML Appendix — "What Predicts Growth?"** (page 29): a logistic regression reporting 71% holdout accuracy separating growing from declining pages.

*Where does the label come from?* Growth/decline is "calculated from 30d-vs-prev-30d impression change" (page 5) -- the same shape of proxy label this track has used since Week 1.

*Does the validation design carry the claim?* The Methodology page states "Random Forest (80/20 split), Logistic Regression (80/20 split)" with no mention of what the split is grouped by. Given the dataset spans only **57 brands** across 341,701 pages, that matters a lot: if the 80/20 split is a plain random *row* split rather than grouped by brand, pages from the same brand almost certainly land in both train and test. Per Week 5's own finding here, rows from the same client/brand share hidden structure (site design, publishing cadence, topic mix) that a random split lets a model partially memorize. A 71% holdout accuracy could be a mix of real, generalizable signal and brand memorization -- and there's no way to tell which from what's published. This is the exact same split-honesty question I had to answer for my own Week 5 model, which is why I'm asking it here rather than taking the 71% at face value.

Constructive framing, not a takedown: both findings are clearly labeled as directional/exploratory elsewhere in the paper, and the authors already show good instincts (demoting weak findings, flagging the ML appendix as secondary). These are the two places I'd ask for one more sentence of methodology detail before repeating the numbers as strongly as the playbook does.

In [1]:
audited_findings = [
    {
        "finding": "#4 The Freshness Multiplier (3.2x health, 57x impressions)",
        "label_source": "observed group split: 365+ day pages refreshed within 30d vs not",
        "open_question": "paired before/after per page, or cross-sectional (selection bias risk)?",
    },
    {
        "finding": "ML Appendix: What Predicts Growth? (71% holdout accuracy)",
        "label_source": "30d-vs-prev-30d impression change (proxy label, same shape as Week 1-5)",
        "open_question": "was the 80/20 split grouped by brand, or a plain random row split (57 brands only)?",
    },
]
for f in audited_findings:
    print(f["finding"])
    print(f"  label source: {f['label_source']}")
    print(f"  open question: {f['open_question']}\n")

#4 The Freshness Multiplier (3.2x health, 57x impressions)
  label source: observed group split: 365+ day pages refreshed within 30d vs not
  open question: paired before/after per page, or cross-sectional (selection bias risk)?

ML Appendix: What Predicts Growth? (71% holdout accuracy)
  label source: 30d-vs-prev-30d impression change (proxy label, same shape as Week 1-5)
  open question: was the 80/20 split grouped by brand, or a plain random row split (57 brands only)?



## 2. My model under an honest split (before/after)

Rebuild the exact Week 5 modeling frame (same query, same leakage fix), then compare Random Forest under two splits on the *same data*: a naive **random row split** (the dishonest "before") against the **client-holdout split** already used in Week 5 (the honest "after"). Per the skill: "the GAP between them is itself a finding about how much memorization was happening" -- this is a direct test of the exact question I just raised about the paper's own unstated split strategy above.

In [2]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

token = os.environ["HF_TOKEN"]
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")

MONTH = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

features_h1 = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impressions_h1, SUM(gsc_clicks) AS clicks_h1,
        AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_h1,
        SUM(ga4_sessions) AS sessions_h1, SUM(ga4_engaged_sessions) AS engaged_sessions_h1
    FROM {MONTH} WHERE report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()
target_h2 = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_h2
    FROM {MONTH} WHERE report_date > DATE '2026-03-15' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()
content_meta = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        DATE_DIFF('day', content_updated_date, DATE '2026-03-15') AS days_since_last_update,
        DATE_DIFF('day', content_created_date, DATE '2026-03-15') AS content_age_days,
        word_count
    FROM {CONTENT}
""").df()

data = (features_h1.merge(target_h2, on=["client_hash_id", "content_hash_id"], how="inner")
                    .merge(content_meta, on=["client_hash_id", "content_hash_id"], how="left"))
data = data[data["impressions_h1"] >= 50].copy()
data = data[(data["days_since_last_update"] >= 0) & (data["content_age_days"] >= 0)].copy()  # Week 5's leakage fix
data["is_declining_label"] = (data["impressions_h2"] < data["impressions_h1"]).astype(int)
data = data.fillna(0)
data = data.sort_values("content_hash_id").reset_index(drop=True)  # Week 5's reproducibility fix

FEATURES = ["impressions_h1", "clicks_h1", "avg_position_h1", "sessions_h1",
            "engaged_sessions_h1", "days_since_last_update", "content_age_days", "word_count"]

def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    if (scores > 0).sum() == 0:
        return None
    order = np.argsort(-scores, kind="stable")
    return np.asarray(labels)[order[:k]].mean()

# --- BEFORE: naive random ROW split (the dishonest baseline) ---
X, y = data[FEATURES], data["is_declining_label"].values
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, class_weight="balanced").fit(X_tr_r, y_tr_r)
score_random = rf_random.predict_proba(X_te_r)[:, 1]

# --- AFTER: client-holdout split (Week 5's honest split, rebuilt identically) ---
clients = sorted(data["client_hash_id"].unique().tolist())
rng = np.random.default_rng(42)
rng.shuffle(clients)
n_test_clients = max(1, round(len(clients) * 0.2))
test_clients = set(clients[:n_test_clients])
train_clients = set(clients[n_test_clients:])
train_df = data[data["client_hash_id"].isin(train_clients)]
test_df = data[data["client_hash_id"].isin(test_clients)]
X_tr_c, y_tr_c = train_df[FEATURES], train_df["is_declining_label"].values
X_te_c, y_te_c = test_df[FEATURES], test_df["is_declining_label"].values
rf_client = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, class_weight="balanced").fit(X_tr_c, y_tr_c)
score_client = rf_client.predict_proba(X_te_c)[:, 1]

before_p50 = precision_at_k(score_random, y_te_r, 50)
after_p50 = precision_at_k(score_client, y_te_c, 50)
print(f"BEFORE (random row split, {len(X_te_r):,} test rows, base rate {y_te_r.mean():.3f}):  P@50 = {before_p50:.3f}")
print(f"AFTER  (client-holdout,   {len(X_te_c):,} test rows, base rate {y_te_c.mean():.3f}):  P@50 = {after_p50:.3f}")
print(f"\nGap: {before_p50 - after_p50:+.3f}")
if before_p50 > after_p50:
    print("The random split looks better -- some of that gap is memorization the honest split removes.")
else:
    print("The random split does NOT look better here -- no evidence of memorization inflating the naive split.")

BEFORE (random row split, 3,525 test rows, base rate 0.550):  P@50 = 0.840
AFTER  (client-holdout,   1,979 test rows, base rate 0.356):  P@50 = 0.640

Gap: +0.200
The random split looks better -- some of that gap is memorization the honest split removes.


## 3. Leakage audit

The same hunt from Week 3 (`w03_data_contract.ipynb`), re-run against Week 5's *final* 8-feature set, plus the attack checklist and one new discovery from actually doing this audit.

In [3]:
from sklearn.metrics import roc_auc_score

checklist = {
    "Timeline drawn: all features strictly before the label window": "YES -- all 8 features aggregated Mar 1-15 only, label from Mar 16-31 only (see Section 2 query)",
    "No label-derived or sibling columns in features": "checked live below",
    "No product flags / existing-system scores as features": "YES -- none exist in this warehouse release by design (lane guide Section 4)",
    "Population selection checked for outcome-window information": "NEW FINDING below -- see discussion",
    "Split grouped by the repeating entity": "YES -- client-holdout, verified 0 client overlap in Week 5 and Section 2 above",
    "Base rate printed next to every metric": "YES -- 0.356 printed in Week 5 and Section 2",
    "Top feature importance sanity-checked": "YES -- avg_position_h1 (0.244) top in Week 5, not suspiciously dominant",
    "Metrics recomputed out-of-fold, never in-sample": "YES -- all Precision@K numbers are on held-out test rows only",
    "Sealed/holdout claims: frame-builder + metrics committed": "YES -- w05_model.ipynb + w05_model_metrics.json both committed",
}
for item, status in checklist.items():
    print(f"[{'x' if not status.startswith('NEW') and not status.startswith('checked') else ' '}] {item}")
    print(f"      {status}")

# --- Live re-test: deliberately ADD a label-derived feature and watch the score jump ---
X_honest = data[FEATURES]
y_all = data["is_declining_label"].values
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(X_honest, y_all, test_size=0.2, random_state=42, stratify=y_all)
honest_model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(Xh_tr, yh_tr)
honest_auc = roc_auc_score(yh_te, honest_model.predict_proba(Xh_te)[:, 1])

data_leak = data.copy()
data_leak["LEAK_impressions_h2"] = data_leak["impressions_h2"]
X_leaky = data_leak[FEATURES + ["LEAK_impressions_h2"]]
Xl_tr, Xl_te, _, _ = train_test_split(X_leaky, y_all, test_size=0.2, random_state=42, stratify=y_all)
leaky_model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(Xl_tr, yh_tr)
leaky_auc = roc_auc_score(yh_te, leaky_model.predict_proba(Xl_te)[:, 1])

print(f"\nLive re-test on Week 5's FINAL 8-feature set:")
print(f"  Honest AUC (8 features):                {honest_auc:.3f}")
print(f"  Leaky AUC (8 features + impressions_h2): {leaky_auc:.3f}  <- confirms the test harness catches it")
print(f"  My test harness is working: {'YES' if leaky_auc > honest_auc + 0.15 else 'NO -- investigate'}")

[x] Timeline drawn: all features strictly before the label window
      YES -- all 8 features aggregated Mar 1-15 only, label from Mar 16-31 only (see Section 2 query)
[ ] No label-derived or sibling columns in features
      checked live below
[x] No product flags / existing-system scores as features
      YES -- none exist in this warehouse release by design (lane guide Section 4)
[ ] Population selection checked for outcome-window information
      NEW FINDING below -- see discussion
[x] Split grouped by the repeating entity
      YES -- client-holdout, verified 0 client overlap in Week 5 and Section 2 above
[x] Base rate printed next to every metric
      YES -- 0.356 printed in Week 5 and Section 2
[x] Top feature importance sanity-checked
      YES -- avg_position_h1 (0.244) top in Week 5, not suspiciously dominant
[x] Metrics recomputed out-of-fold, never in-sample
      YES -- all Precision@K numbers are on held-out test rows only
[x] Sealed/holdout claims: frame-builder + metr


Live re-test on Week 5's FINAL 8-feature set:
  Honest AUC (8 features):                0.685
  Leaky AUC (8 features + impressions_h2): 0.936  <- confirms the test harness catches it
  My test harness is working: YES


**The population-selection question, actually checked:** Week 5's leakage fix dropped any row where `content_updated_date` fell after March 15 (the "future-dated" rows `dim_content` couldn't vouch for). That fix was correct for leakage, but it also has a side effect worth naming: the surviving population is, by construction, **pages that did NOT get edited between March 15 and July**. If editors are more likely to refresh pages they already believe are declining (the same selection-bias question raised in Section 1 against the paper's own Finding #4), then my modeling population may be quietly *under-representing* the declining pages editors would have already caught and fixed. This isn't leakage in the classic sense -- no future information reaches a feature -- but it is a population choice shaped by the outcome-adjacent behavior of editors, and it should be disclosed rather than left implicit. Flagging it here as a named limitation for the capstone paper, not fixing it now: fixing it properly would need point-in-time content history, which `dim_content` doesn't provide.

## 4. Claim rewrite

**My boldest sentence, from Week 5's Section 4:** "Random Forest wins at P@50 (0.64 vs 0.46) but loses at P@20 (0.60 vs 0.65)." Read on its own, "wins" sounds like a settled, general fact about the model.

**Rewritten in safe language:** *On this run's client-holdout test set (1,979 rows, 6 held-out clients), Random Forest was observed to score higher than Logistic Regression at Precision@50 (0.64 vs. 0.46) and lower at Precision@20 (0.60 vs. 0.65). Both directional results come from a single train/test split on one month of data; Section 2 above shows a naive random split would have reported different numbers from the same underlying data, which is itself evidence that any one split's numbers are a measurement with real variance, not a fixed property of the model. This is decision-support evidence for preferring Random Forest's precision profile at K=50 specifically, not a general claim that Random Forest is the better model.*

The rewrite doesn't change the number -- it changes what the number is allowed to claim.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.